In [0]:
dbutils.widgets.dropdown(name = "environment", defaultValue= "dev",choices= ["dev","prod","qa"],label = "select Environment")
env = dbutils.widgets.get("environment")
# print(env)


silverTablName = f"saleslake_{env}.silver_{env}.cleanedsales"
print("silverTablName:", silverTablName)
goldTablName = f"saleslake_{env}.gold_{env}.refinedsales"
print("goldTablName:",goldTablName )

In [0]:
spark.sql(f"""
MERGE INTO {goldTablName} tgt
USING (
    WITH latest_inv_silver AS (
        -- Step 1: Filter only new/updated silver records since last gold load
        SELECT *
        FROM {silverTablName}
        WHERE ingest_ts > (
            SELECT COALESCE(MAX(last_updt_ts), TO_TIMESTAMP('1990-01-01', 'yyyy-MM-dd'))
            FROM {goldTablName}
        )
    ),

    latest_rm_dup_silver AS (
        -- Step 2: Deduplicate — keep latest record per sale_id
        SELECT * FROM (
            SELECT *,
                   ROW_NUMBER() OVER (PARTITION BY sale_id ORDER BY ingest_ts DESC) AS rn
            FROM latest_inv_silver
        ) WHERE rn = 1
    )

    -- Step 3: Select fields matching gold table schema
    SELECT
        sale_id,
        CAST(product_id AS STRING) AS product,
        CAST(NULL AS STRING) AS category,
        quantity,
        CAST(unit_price AS DOUBLE) AS price,
        sale_date,
        CAST(region_id AS STRING) AS region,
        ingest_ts AS initial_load_ts
    FROM latest_rm_dup_silver

) src
ON tgt.sale_id = src.sale_id

-- Update existing records
WHEN MATCHED THEN UPDATE SET
    tgt.product = src.product,
    tgt.category = src.category,
    tgt.quantity = src.quantity,
    tgt.price = src.price,
    tgt.sale_date = src.sale_date,
    tgt.region = src.region,
    tgt.last_updt_ts = CURRENT_TIMESTAMP()

-- Insert new records
WHEN NOT MATCHED THEN INSERT (
    sale_id, product, category, quantity, price, sale_date, region, initial_load_ts, last_updt_ts
)
VALUES (
    src.sale_id, src.product, src.category, src.quantity, src.price, src.sale_date, src.region, src.initial_load_ts, CURRENT_TIMESTAMP()
)
""")

In [0]:
%sql
SELECT count(*) FROM saleslake_dev.gold_dev.refinedsales